### 1. What is your model structure?
I am using `distilbert-base-uncased`, a pre-trained Transformer language model from Hugging Face. It is a smaller, faster, and lighter version of BERT. I added a sequence classification head on top of it with 2 output labels (Negative=0, Positive=1).

### 2. How did you handle the small and imbalanced training set?
**Class Imbalance:** The training set is heavily skewed (180 positive, 60 negative). I handled this by oversampling the minority class (negative reviews) using `pandas` and `scikit-learn`'s resample tool so the model trains on a 50/50 split.
**Small Training Set & Overfitting:** To prevent overfitting on such a tiny dataset, I utilized transfer learning (a pre-trained model) rather than training from scratch. I kept the training short (only 3 epochs) and applied weight decay (L2 regularization). 
**Unseen Tokens:** Because DistilBERT uses subword tokenization (WordPiece), it gracefully handles evaluation tokens that do not appear in the training data by breaking them down into known subwords.

### 3. Key training techniques
- **Optimizer & Learning Rate:** Used AdamW optimizer with a low learning rate of `2e-5` to gently fine-tune the pre-trained weights.
- **Batch Size:** `8` to ensure memory efficiency on standard CPUs while providing stable gradients.
- **Weight Decay:** `0.01` to penalize large weights and further prevent overfitting.

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Load the datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('public_test.csv')

print("Original Training Distribution:")
print(train_df['label'].value_counts())

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original Training Distribution:
label
1    180
0     60
Name: count, dtype: int64


In [2]:
# Separate majority and minority classes
df_majority = train_df[train_df.label == 1]
df_minority = train_df[train_df.label == 0]

# Upsample minority class
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=180,    # match majority class size
                                 random_state=42)  # reproducible results

# Combine and shuffle
train_df_balanced = pd.concat([df_majority, df_minority_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced Training Distribution:")
print(train_df_balanced['label'].value_counts())


Balanced Training Distribution:
label
0    180
1    180
Name: count, dtype: int64


In [3]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert Pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df_balanced)
test_ds = Dataset.from_pandas(test_df)

# Tokenize function
def tokenize_func(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

train_encoded = train_ds.map(tokenize_func, batched=True)
test_encoded = test_ds.map(tokenize_func, batched=True)

Map: 100%|██████████| 400/400 [00:00<00:00, 579.58 examples/s]


In [4]:
# Load pre-trained model with 2 classification labels
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="no", # Do not save intermediate checkpoints to save space
    use_cpu=True        # Ensures it runs smoothly on your Codespace CPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
)

# Start training
trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1448.81it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.697418
20,0.691002
30,0.694278
40,0.665873
50,0.662334
60,0.607166
70,0.566138
80,0.524397
90,0.514487
100,0.426921


TrainOutput(global_step=135, training_loss=0.5462657919636479, metrics={'train_runtime': 599.9643, 'train_samples_per_second': 1.8, 'train_steps_per_second': 0.225, 'total_flos': 35766197637120.0, 'train_loss': 0.5462657919636479, 'epoch': 3.0})

In [5]:
# Get predictions
predictions = trainer.predict(test_encoded)
predicted_labels = np.argmax(predictions.predictions, axis=-1)

# Calculate accuracy and confusion matrix
acc = accuracy_score(test_df['label'], predicted_labels)
cm = confusion_matrix(test_df['label'], predicted_labels)

print(f"Total Accuracy on Public Test Data: {acc * 100:.2f}%")
print(f"Confusion Matrix:\n{cm}")

Total Accuracy on Public Test Data: 58.75%
Confusion Matrix:
[[ 63 137]
 [ 28 172]]


In [6]:
# 1. Save the model and tokenizer to model_checkpoint/ folder
model.save_pretrained("./model_checkpoint")
tokenizer.save_pretrained("./model_checkpoint")
print("Model saved to ./model_checkpoint/")

# 2. Create the required public_test_predictions.csv
test_df['predicted_label'] = predicted_labels

# Only keep id and predicted_label columns
submission_df = test_df[['id', 'predicted_label']]
submission_df.to_csv('public_test_predictions.csv', index=False)
print("Saved predictions to public_test_predictions.csv")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]

Model saved to ./model_checkpoint/
Saved predictions to public_test_predictions.csv
